# Etapa 01: El problema y la base de muestras

En esta etapa contamos qué pregunta queríamos responder, qué columnas esperamos de la base del laboratorio
(y en qué momento del proceso se conoce cada una) y dejamos guardadas las tablas crudas tal como llegan, sin
limpiar nada todavía.

[Notebook completo](Proyecto_Final.ipynb)<br>Siguiente: [Etapa 02: Exploración](02_exploracion.ipynb) →

## Código del proyecto

La celda siguiente vuelve a crear el paquete `proyecto/` dentro de la sesión. Así el notebook funciona en Google Colab sin clonar ningún repositorio: Basta con ejecutarla una vez (*Entorno de ejecución → Ejecutar todas*). En una copia local del repositorio no reemplaza ningún archivo y la sección se puede plegar desde su título.

In [ ]:
# Código del proyecto. Esta celda recrea en el directorio de trabajo:
#   - el paquete `proyecto/` (55 módulos) y `configuracion/mapeo_columnas.json`;
# para que el notebook funcione en Google Colab sin clonar el repositorio. Todo viaja
# al final de la celda como un .tar.xz en base64 (66 archivos, 206 kB).
# Antes de extraer se comprueba su huella SHA-256. Un archivo que ya existe con el
# mismo contenido no se toca, y uno que existe con otro contenido (una copia local
# del repositorio con cambios, por ejemplo) se conserva y se avisa; para
# reemplazarlo hay que poner SOBRESCRIBIR = True.
import base64
import hashlib
import importlib
import io
import os
import sys
import tarfile
from pathlib import Path

SOBRESCRIBIR = False


def recrear_proyecto(huella, codificado):
    datos = base64.b64decode(codificado)
    if hashlib.sha256(datos).hexdigest() != huella:
        raise ValueError("El código embebido está dañado: su SHA-256 no coincide.")
    escritos, iguales, conservados = 0, 0, []
    with tarfile.open(fileobj=io.BytesIO(datos), mode="r:xz") as archivo:
        for miembro in archivo.getmembers():
            nombre = miembro.name
            if Path(nombre).is_absolute() or ".." in Path(nombre).parts:
                raise ValueError(f"Ruta no permitida en el archivo: {nombre}")
            contenido, destino = archivo.extractfile(miembro).read(), Path(nombre)
            if destino.exists() and destino.read_bytes() == contenido:
                iguales += 1
            elif destino.exists() and not SOBRESCRIBIR:
                conservados.append(nombre)
            else:
                destino.parent.mkdir(parents=True, exist_ok=True)
                destino.write_bytes(contenido)
                escritos += 1
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())
    importlib.invalidate_caches()
    print(f"Código del proyecto: {escritos} archivos escritos, {iguales} ya estaban iguales.")
    if conservados:
        print(f"Se conservaron {len(conservados)} archivos locales distintos:", ", ".join(conservados))


recrear_proyecto(
    huella="357d1318af567faa08a0a3186d07ebb649d2028b91445d6980362e5387813ac5",
    codificado="...",
)

## Desarrollo de la etapa

In [1]:
from proyecto.base.artefactos import iniciar_etapa
from proyecto.base.salida import mostrar
from proyecto.etapas import datos

iniciar_etapa("01_datos")
c = datos.iniciar()
mostrar(c.encabezado, c.autores)

## 1. El problema

Un laboratorio de microbiología recibe todos los días muestras de producto terminado, materia prima, agua y
muestreos en planta de clientes cosméticos, farmacéuticos, de alimentos, veterinarios y de centrales de mezclas.
De cada muestra guarda quién la envió, cómo llegó, qué ensayo se le hizo, quién la procesó y qué resultado dio. El
contexto es el de Rodam Análisis SAS en Bogotá, aunque lo que estudiamos es la base de muestras y no la empresa.
Con esa base nos propusimos tres preguntas:

a. **Demanda.** Cuántas muestras van a llegar, para programar analistas, medios de cultivo e incubadoras.
b. **Riesgo.** Qué tan probable es que una muestra salga no conforme (fuera del límite que exige la norma) y, si
   sale positiva, qué tan probable es que sea un falso positivo.
c. **Grupos.** Qué clientes o muestras se parecen entre sí.

De cada pregunta salían varios análisis posibles y no nos daba el tiempo para estudiarlos todos a fondo. Por eso
armamos un catálogo de candidatos, los pasamos por la misma prueba rápida en la etapa 03 y solo a los ganadores
los modelamos en serio.

## 2. El contrato de datos

Antes de tener datos escribimos un contrato (`proyecto/datos/esquema.py`), que es el único lugar del código
donde aparecen los nombres de las columnas. De cada columna dice su tipo, su papel, si es obligatoria y el
**momento** del proceso en que el dato pasa a conocerse (la toma, la llegada, la recepción, la lectura...). Si
la exportación real no trae una columna opcional, los análisis que la usan simplemente se omiten.

In [2]:
datos.contrato(c).mostrar()

Columna,Tipo,Papel,Momento,Obligatoria,Descripción
id_cliente,id,id,maestro,sí,Identificador del cliente
sector,categoria,variable,maestro,sí,Línea de negocio del cliente
zona,categoria,variable,maestro,no,Municipio o zona de la planta
tamano,categoria,variable,maestro,no,Tamaño de la empresa
tipo_contrato,categoria,variable,maestro,no,Contrato marco o servicio por demanda
certificacion_bpm,categoria,variable,maestro,no,Cuenta con certificación BPM (sí/no)
plan_muestreo,categoria,variable,maestro,no,Frecuencia pactada de muestreo en planta
fecha_vinculacion,fecha,fecha,maestro,no,Fecha de vinculación como cliente
distancia_km,numero,variable,maestro,no,Distancia de la planta al laboratorio (km)
canal_captacion,categoria,variable,maestro,no,Canal por el que llegó el cliente


El momento es lo que más adelante nos protege de la fuga de información, que es usar para predecir un dato que
todavía no existe en el instante de la predicción. Cada análisis declara en qué momento predice y solo puede
usar las columnas conocidas hasta ahí. Así se lo preguntamos al contrato para una predicción en la recepción:

In [3]:
from proyecto.datos import esquema
from proyecto.reporte.html import fichas

al_recibir = esquema.columnas("muestras", hasta="recepcion")   # lo que ya se sabe cuando la muestra llega
despues = [x for x in esquema.columnas("muestras") if x not in al_recibir]
mostrar(fichas(al_recibir, f"{len(al_recibir)} columnas de muestras que se conocen al recibirla"),
        fichas(despues, f"{len(despues)} que llegan después y quedan prohibidas para esa predicción", color="rojo"))

## 3. Las tablas crudas

Obtenemos las tablas, las validamos contra el contrato (si falta una columna obligatoria la etapa se detiene y
dice cuáles) y las guardamos tal como llegan, en CSV. Así la limpieza de la etapa 02 queda como un paso aparte
que se puede auditar.

In [4]:
datos.tablas_crudas(c).mostrar()

Bastó con mirar las tablas para ver que hacía falta una etapa de limpieza: La misma categoría escrita de varias
formas, recuentos reportados como texto, valores imposibles y filas repetidas.

In [5]:
datos.como_llegan(c).mostrar()

## 4. Los datos simulados y la exportación real

Daniel, del equipo, va a conseguir la exportación real del laboratorio, pero no la teníamos a tiempo para
construir el proyecto. Mientras tanto trabajamos con una base simulada que reproduce el recorrido de cada muestra
por el laboratorio, con la misma suciedad que esperamos encontrar. Las variables ocultas que usa el simulador (si
la muestra estaba de verdad contaminada, qué tan higiénico es cada cliente) quedan aparte en `datos/auditoria/` y
ningún modelo las lee. Cuando llegue la exportación bastaría copiarla en `datos/reales/`, describir en
`configuracion/mapeo_columnas.json` cómo se llama cada columna y cada etiqueta y fijar `PROYECTO_ORIGEN=real`, sin
tocar el código. Eso sí, todas las cifras de estos notebooks describen la base simulada y el método, no todavía
al laboratorio.

In [6]:
datos.exportacion_real(c).mostrar()

Tabla de auditoría,Filas,Columnas latentes
verdad_muestras,38846,"contaminada_real, prob_contaminacion, prob_falso_positivo, positivo_sin_ruido, en_episodio, logit_contaminacion"
verdad_clientes,220,"segmento_latente, higiene, tasa_semanal, fecha_retiro"
verdad_ensayos,7,"ensayo, sensibilidad, especificidad_nominal, especificidad_efectiva, prevalencia_real"


## 5. Protocolo de evaluación y mapa del proyecto

Partimos la base por tiempo con cortes que fijamos en `config` antes de ver un solo resultado. El
**entrenamiento** sirve para ajustar y comparar modelos (con validación cruzada de ventana creciente, que
explicamos en la etapa 02), la **validación** para decidir (el umbral, el modelo que se congela) y la **prueba**
se guarda aparte y solo se abre una vez, en la etapa 07. Las etapas se pasan la información únicamente por
archivos en disco.

In [7]:
datos.protocolo(c).mostrar()

In [8]:
datos.terminar(c)

[Notebook completo](Proyecto_Final.ipynb)<br>Siguiente: [Etapa 02: Exploración](02_exploracion.ipynb) →